# SmolVLA × LIBERO-plus  Track 1 向け fine-tuning（衝突ルール対応）

PARC 2026 Track 1 の成功判定は **ゴール達成 かつ 対象外物体の変位が全ステップで
1mm 以下**という、極端に厳しいものである。素の behavior cloning（既存の Spatial
ノートブック）はこの制約を一切見ずにデモを模倣するだけなので、本番スコアと乖離する。

**正直な技術的注記:** SmolVLA は flow-matching 損失（予測速度 vs 目標速度の MSE）で
学習される。損失の内部に「モデルが出力する行動チャンク」が展開されていないため、
損失へ「衝突ペナルティ項」をきれいに差し込むことはできない（GT 行動に滑らかさ項を
足しても勾配が出ず無意味）。したがって 1mm ルールに効かせる正攻法は、損失いじりでは
なく次の3本柱である。

1. **Track 1 分布に寄せた広いデータで学習する** — 既存ノートは Spatial 10 タスクのみ。
   本ノートは libero_plus（テクスチャ/照明などの摂動を含む）から object/goal も含めて
   広く選び、分布シフト由来の衝突を減らす。
2. **本番の 1mm ルールでチェックポイントを選抜する** — LeRobot の `lerobot-eval` は
   衝突判定をしない＝真のスコアが見えない。学習後は配布キットの
   `python -m pipeline`／`tune.py` で **実際の 1mm 判定つき成功率**を測って最良を選ぶ。
3. **アンサンブル用に複数モデルを量産する** — 設定を変えて数体作り、提出テンプレートの
   `model_weights/<名前>/` に並べると、`policy_server.py` の MyPolicy が全モデルを
   平均して推論する（不確実性が大きい＝モデルが割れる難所ほど自動で慎重に動く）。

出力は LeRobot 形式のマージ済みモデル一式（zip）。これを提出テンプレートの
`model_weights/` に置いて使う。衝突を抑える**推論時の制御（時間方向アンサンブル・
不確実性ダンピング・EMA 等）は `policy_server.py` 側に実装済み**であり、本ノートの
役割は「その土台となる強いベースポリシーを作ること」である。

> ランタイムを GPU（T4 で可）にしてから上から順に実行する。所要時間は設定（STEPS /
> タスク数 / エピソード数）に比例する。まず小さめで一周し、キットで実測してから
> 規模を上げるとよい。


## 1. Colabランタイムを確認する

ColabのランタイムをGPUへ変更してから実行してください。

In [ ]:
import importlib
import importlib.metadata
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["DIFFUSERS_VERBOSITY"] = "error"
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["HF_LEROBOT_HOME"] = "/content/lerobot_cache"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if sys.version_info < (3, 12):
    raise RuntimeError("Python 3.12以上が必要です。")

if not torch.cuda.is_available():
    raise RuntimeError("GPUランタイムを選択してください。")

print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1.5 Google Drive をマウントする（スマホ運用向け）

学習後のモデルを **自動で Google Drive に保存**する。ブラウザのダウンロードに
頼らないため、スマホでアプリを切り替えても結果が失われにくい。

保存先は `DRIVE_SAVE_DIR` にまとめて置かれるので、Colab を複数回実行しても
過去の結果は残る（`MODEL_TAG` ごとにサブフォルダが分かれる）。


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_SAVE_ROOT = Path('/content/drive/MyDrive/parc2026_models')
    DRIVE_SAVE_ROOT.mkdir(parents=True, exist_ok=True)
    _HAS_DRIVE = True
    print(f"Google Drive mounted. 保存先: {DRIVE_SAVE_ROOT}")
except Exception as exc:  # noqa: BLE001 - Colab以外(ローカル等)では無視
    DRIVE_SAVE_ROOT = None
    _HAS_DRIVE = False
    print(f"Google Drive はマウントしません（Colab外、または失敗: {exc}）")


## 2. システムパッケージを準備する

LeRobot、動画デコード、MuJoCoで必要になるパッケージを導入します。

In [ ]:
def run_quiet(
    command: list[str],
    *,
    check: bool = True,
) -> subprocess.CompletedProcess:
    result = subprocess.run(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )

    if check and result.returncode != 0:
        raise RuntimeError(result.stdout[-6000:])

    return result


run_quiet(["apt-get", "update", "-qq"])
run_quiet(
    [
        "apt-get",
        "install",
        "-y",
        "-qq",
        "ffmpeg",
        "git",
        "unzip",
        "libgl1",
        "libglib2.0-0",
        "libsm6",
        "libxext6",
        "libexpat1",
        "libfontconfig1-dev",
        "libmagickwand-dev",
    ]
)

print("System packages ready.")

## 3. LeRobotをインストールする

LeRobot `v0.6.0`を使用します。
ColabでのLoRA学習に必要な互換性調整もこのセルで適用します。

In [ ]:
LEROBOT_TAG = "v0.6.0"
LEROBOT_DIR = Path("/content/lerobot")
LEROBOT_SRC = LEROBOT_DIR / "src"

run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "lerobot",
        "torchao",
    ],
    check=False,
)

shutil.rmtree(LEROBOT_DIR, ignore_errors=True)

run_quiet(
    [
        "git",
        "clone",
        "--quiet",
        "--depth",
        "1",
        "--branch",
        LEROBOT_TAG,
        "https://github.com/huggingface/lerobot.git",
        str(LEROBOT_DIR),
    ]
)

smolvlm_source = (
    LEROBOT_SRC
    / "lerobot"
    / "policies"
    / "smolvla"
    / "smolvlm_with_expert.py"
)

if not torch.cuda.is_bf16_supported():
    source = smolvlm_source.read_text(encoding="utf-8")
    source = source.replace(
        'torch_dtype="bfloat16",',
        'torch_dtype="float16",',
        1,
    )
    smolvlm_source.write_text(
        source,
        encoding="utf-8",
    )

train_script = (
    LEROBOT_SRC
    / "lerobot"
    / "scripts"
    / "lerobot_train.py"
)
source = train_script.read_text(encoding="utf-8")
source = source.replace(
    "logging.info(pformat(cfg.to_dict()))",
    "logging.debug(pformat(cfg.to_dict()))",
    1,
)
source = source.replace(
    "disable=inside_slurm(),",
    "disable=True,",
    1,
)
train_script.write_text(
    source,
    encoding="utf-8",
)

run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--upgrade",
        "-e",
        f"{LEROBOT_DIR}[training,smolvla,peft]",
    ]
)

run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "torchao",
    ],
    check=False,
)

for module_name in list(sys.modules):
    if (
        module_name == "lerobot"
        or module_name.startswith("lerobot.")
        or module_name == "torchao"
        or module_name.startswith("torchao.")
    ):
        del sys.modules[module_name]

sys.path = [
    item
    for item in sys.path
    if item not in {
        str(LEROBOT_DIR),
        str(LEROBOT_SRC),
    }
]
sys.path.insert(0, str(LEROBOT_SRC))
importlib.invalidate_caches()

try:
    importlib.metadata.version("torchao")
except importlib.metadata.PackageNotFoundError:
    pass
else:
    raise RuntimeError("torchaoの削除に失敗しました。")

import lerobot
import peft

if (
    LEROBOT_SRC.resolve()
    not in Path(lerobot.__file__).resolve().parents
):
    raise RuntimeError("LeRobotの読込先が正しくありません。")

print("LeRobot ready.")

## 学習・評価条件を設定する

Track 1 に寄せて、Spatial だけでなく libero_plus 全体から広くタスクを選ぶ。まず小さめ（STEPS/タスク数を下げる）で 一周し、キットで実測してから規模を上げるとよい。

In [ ]:
# === 学習条件（Track 1 に寄せた広めの設定）===
BASE_MODEL_REPO = "lerobot/smolvla_libero_plus"
BASE_MODEL_REVISION = "7bb70aa5bc92b82c9239142775d3a173103567ff"
VLM_REPO = "HuggingFaceTB/SmolVLM2-500M-Video-Instruct"
DATASET_REPO = "lerobot/libero_plus"
DATASET_REVISION = "f3f49f426d75030177b18778374005bc12ccd588"

# --- どのタスクを学習するか ---
# TASK_NAME_FILTERS に部分文字列を入れると、その語を含むタスクだけを選ぶ。
# 空リスト = libero_plus の全タスクから選ぶ。Track 1 は spatial/object/goal を
# 含むため、既定では絞らず広く学習する。
# アンサンブルの専門家を作る例:
#   ["bowl"]               # bowl 系に特化
#   ["basket"]             # basket 系に特化
TASK_NAME_FILTERS: list[str] = []
TRAIN_EPISODES_PER_TASK = 3      # 1 タスクあたりの学習エピソード数
MAX_TASKS = 40                   # 学習に使うタスク数の上限（None で全タスク）

# --- 学習ハイパーパラメータ ---
STEPS = 6000
LOG_FREQ = 100
BATCH_SIZE = 1
LEARNING_RATE = 3e-4
FINAL_LEARNING_RATE = 3e-5
WARMUP_STEPS = 200
LORA_R = 16
LORA_ALPHA = 16
SEED = 42

# --- アンサンブル用のモデル名（提出時に model_weights/<MODEL_TAG>/ に置く想定）---
# 別の専門家を作るときは MODEL_TAG と TASK_NAME_FILTERS / SEED を変えて再実行する。
MODEL_TAG = "track1_lora"

OUTPUT_DIR = Path(f"/content/outputs/smolvla_{MODEL_TAG}")
MERGED_MODEL_DIR = Path(f"/content/smolvla_{MODEL_TAG}_merged")
MERGED_ZIP_PATH = Path(f"/content/smolvla_{MODEL_TAG}_merged.zip")

# Drive 保存先（1.5 のセルでマウント済みなら自動で使われる）
DRIVE_MODEL_DIR = (
    DRIVE_SAVE_ROOT / MODEL_TAG if _HAS_DRIVE else None
)

MIXED_PRECISION = "bf16" if torch.cuda.is_bf16_supported() else "fp16"

# --- チェックポイント保存間隔 ---
# 途中で接続が切れても学習をやり直さずに済むよう、最終ステップだけでなく
# 一定間隔でも保存する（スマホはバックグラウンドタブが切れやすいため必須）。
SAVE_FREQ = max(1, min(STEPS, 500))

print(f"MODEL_TAG={MODEL_TAG}  STEPS={STEPS}  SAVE_FREQ={SAVE_FREQ}  "
      f"episodes/task={TRAIN_EPISODES_PER_TASK}  max_tasks={MAX_TASKS}")
if _HAS_DRIVE:
    print(f"Drive 保存先: {DRIVE_MODEL_DIR}")

# --- スマホでこのセッションを見続けられる時間の目安 ---
# モバイルブラウザは他アプリに切り替えるとタブがサスペンドされ、WebSocket が
# 切れて学習が止まることがある。画面を見続けられる時間が短い場合は、
# 上のセルで STEPS を下げて一回で完走できる規模にすること（目安: T4 で
# STEPS=1500〜2000, MAX_TASKS=10〜15 なら数十分程度で完走する）。


## 5. 公開ファイルの取得処理を用意する

キャッシュを優先し、匿名アクセスの制限時は自動的に再試行します。

In [ ]:
import random
import time
from collections.abc import Callable
from typing import TypeVar

import httpx
from huggingface_hub import snapshot_download
from huggingface_hub.errors import (
    HfHubHTTPError,
    LocalEntryNotFoundError,
)

T = TypeVar("T")


def run_hf_with_retry(
    operation: Callable[[], T],
) -> T:
    last_error: BaseException | None = None

    for attempt in range(6):
        try:
            return operation()
        except (
            HfHubHTTPError,
            httpx.HTTPStatusError,
        ) as error:
            last_error = error
            response = getattr(error, "response", None)
            status = getattr(response, "status_code", None)

            if status != 429 and "429" not in str(error):
                raise

            if attempt == 5:
                break

            headers = getattr(response, "headers", {}) or {}
            try:
                delay = float(
                    headers.get("Retry-After", 15)
                ) + 1
            except (TypeError, ValueError):
                delay = min(
                    120,
                    15 * (2**attempt) + random.random(),
                )

            time.sleep(delay)

    raise RuntimeError(
        "Hugging Faceからの取得に失敗しました。"
    ) from last_error


def cached_or_downloaded_snapshot(
    repo_id: str,
    revision: str,
    *,
    allow_patterns: list[str] | None = None,
    ignore_patterns: list[str] | None = None,
) -> Path:
    try:
        return Path(
            snapshot_download(
                repo_id=repo_id,
                revision=revision,
                token=False,
                allow_patterns=allow_patterns,
                ignore_patterns=ignore_patterns,
                local_files_only=True,
            )
        )
    except (
        LocalEntryNotFoundError,
        FileNotFoundError,
    ):
        return Path(
            run_hf_with_retry(
                lambda: snapshot_download(
                    repo_id=repo_id,
                    revision=revision,
                    token=False,
                    allow_patterns=allow_patterns,
                    ignore_patterns=ignore_patterns,
                    max_workers=1,
                )
            )
        )

## 学習データを選ぶ（全スイートから広く）

`TASK_NAME_FILTERS` が空なら libero_plus の全タスクから、各タスク `TRAIN_EPISODES_PER_TASK` 本ずつを均等サンプルする。

In [ ]:
import re
from collections import defaultdict

from lerobot.datasets.dataset_metadata import LeRobotDatasetMetadata


def normalize_task_name(value: str) -> str:
    value = value.lower().replace("_", " ")
    value = re.sub(r"[^a-z0-9 ]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()


def task_name_from_cell(value) -> str:
    if isinstance(value, str):
        return value
    try:
        if len(value) > 0:
            return str(value[0])
    except TypeError:
        pass
    return str(value)


def choose_evenly_spaced(episode_indices, count):
    n = len(episode_indices)
    count = min(count, n)
    if count <= 0:
        return []
    if count == 1:
        return [episode_indices[n // 2]]
    positions = [round(i * (n - 1) / (count - 1)) for i in range(count)]
    seen, out = set(), []
    for p in positions:                       # 端数丸めの重複を除去
        if p not in seen:
            seen.add(p)
            out.append(episode_indices[p])
    return out


dataset_metadata = run_hf_with_retry(
    lambda: LeRobotDatasetMetadata(DATASET_REPO, revision=DATASET_REVISION)
)

task_to_episodes: dict[str, list[int]] = defaultdict(list)
for episode_index, task_cell in enumerate(dataset_metadata.episodes["tasks"]):
    task_to_episodes[task_name_from_cell(task_cell)].append(int(episode_index))


def keep_task(name: str) -> bool:
    if not TASK_NAME_FILTERS:
        return True
    norm = normalize_task_name(name)
    return any(normalize_task_name(f) in norm for f in TASK_NAME_FILTERS)


candidate_tasks = sorted(t for t in task_to_episodes if keep_task(t))
if MAX_TASKS is not None:
    candidate_tasks = candidate_tasks[:MAX_TASKS]
if not candidate_tasks:
    raise RuntimeError(
        "条件に合うタスクがありません。TASK_NAME_FILTERS を見直してください。"
    )

selected_by_task = {
    t: choose_evenly_spaced(task_to_episodes[t], TRAIN_EPISODES_PER_TASK)
    for t in candidate_tasks
}
EPISODE_INDICES = sorted(
    idx for eps in selected_by_task.values() for idx in eps
)
if not EPISODE_INDICES:
    raise RuntimeError("Episode selection failed.")

print(f"Training data: {len(candidate_tasks)} tasks, "
      f"{len(EPISODE_INDICES)} episodes")
print("最初の5タスク:", candidate_tasks[:5])


## 7. 初期重みを準備する

In [ ]:
BASE_MODEL_LOCAL = cached_or_downloaded_snapshot(
    BASE_MODEL_REPO,
    BASE_MODEL_REVISION,
    allow_patterns=[
        "config.json",
        "model.safetensors",
        "train_config.json",
        "policy_preprocessor.json",
        "policy_preprocessor*.safetensors",
        "policy_postprocessor.json",
        "policy_postprocessor*.safetensors",
    ],
    ignore_patterns=[
        "README.md",
        "eval/**",
    ],
)

if not (
    BASE_MODEL_LOCAL / "model.safetensors"
).is_file():
    raise FileNotFoundError("Base model not found.")

print("Base model ready.")

## 8. LoRA学習を実行する

100 stepごとに平均lossとlearning rateを表示します。

In [ ]:
import re
from collections import deque

episodes_json = (
    "["
    + ",".join(map(str, EPISODE_INDICES))
    + "]"
)

command = [
    "lerobot-train",
    f"--policy.path={BASE_MODEL_LOCAL}",
    f"--policy.vlm_model_name={VLM_REPO}",
    "--policy.push_to_hub=false",
    "--policy.repo_id=null",
    "--policy.input_features=null",
    "--policy.output_features=null",
    "--policy.empty_cameras=0",
    "--policy.freeze_vision_encoder=true",
    "--policy.train_expert_only=true",
    f"--policy.optimizer_lr={LEARNING_RATE}",
    f"--policy.scheduler_decay_lr={FINAL_LEARNING_RATE}",
    f"--policy.scheduler_warmup_steps={WARMUP_STEPS}",
    f"--policy.scheduler_decay_steps={STEPS}",
    f"--dataset.repo_id={DATASET_REPO}",
    f"--dataset.revision={DATASET_REVISION}",
    f"--dataset.episodes={episodes_json}",
    "--dataset.use_imagenet_stats=false",
    "--dataset.video_backend=torchcodec",
    f"--output_dir={OUTPUT_DIR}",
    "--job_name=smolvla_libero_plus_spatial_lora",
    f"--steps={STEPS}",
    f"--batch_size={BATCH_SIZE}",
    "--num_workers=0",
    "--persistent_workers=false",
    "--env_eval_freq=0",
    "--eval_steps=0",
    f"--seed={SEED}",
    "--save_checkpoint=true",
    f"--save_freq={SAVE_FREQ}",
    "--save_checkpoint_to_hub=false",
    f"--log_freq={LOG_FREQ}",
    "--wandb.enable=false",
    "--peft.method_type=LORA",
    f"--peft.r={LORA_R}",
    f"--peft.lora_alpha={LORA_ALPHA}",
]

training_env = os.environ.copy()
training_env["PYTHONPATH"] = (
    str(LEROBOT_SRC)
    + os.pathsep
    + training_env.get("PYTHONPATH", "")
)
training_env["ACCELERATE_MIXED_PRECISION"] = (
    MIXED_PRECISION
)
training_env["PYTHONUNBUFFERED"] = "1"
training_env["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
training_env["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
training_env["HF_HUB_VERBOSITY"] = "error"
training_env["TQDM_DISABLE"] = "1"
training_env["PYTHONWARNINGS"] = "ignore"

shutil.rmtree(OUTPUT_DIR, ignore_errors=True)

print("Preparing data and starting training...")

process = subprocess.Popen(
    command,
    cwd=LEROBOT_DIR,
    env=training_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

recent_lines: deque[str] = deque(maxlen=80)
report_step = LOG_FREQ

assert process.stdout is not None

for raw_line in process.stdout:
    line = raw_line.replace("\r", "").strip()

    if not line:
        continue

    recent_lines.append(line)

    if "step:" in line and "loss:" in line:
        loss_match = re.search(
            r"loss:([0-9.eE+-]+)",
            line,
        )
        lr_match = re.search(
            r"lr:([0-9.eE+-]+)",
            line,
        )

        loss = (
            loss_match.group(1)
            if loss_match
            else "n/a"
        )
        lr = (
            lr_match.group(1)
            if lr_match
            else "n/a"
        )

        print(
            f"step {report_step:4d}/{STEPS}  "
            f"loss={loss}  lr={lr}"
        )
        report_step += LOG_FREQ

return_code = process.wait()

if return_code != 0:
    print("\n".join(recent_lines))
    raise RuntimeError(
        f"Training failed: {return_code}"
    )

print("Training complete.")

## 9. LoRAをマージしてモデル全体を保存する

LoRA差分を元weightへ統合し、通常のLeRobotモデルとして保存します。

In [ ]:
import contextlib
import gc
import io
import json

from peft import PeftModel
from safetensors import safe_open
from lerobot.configs import PreTrainedConfig
from lerobot.policies.smolvla.modeling_smolvla import (
    SmolVLAPolicy,
)

checkpoint_dir = (
    OUTPUT_DIR
    / "checkpoints"
    / f"{STEPS:06d}"
    / "pretrained_model"
)

if not (
    checkpoint_dir / "adapter_model.safetensors"
).is_file():
    raise FileNotFoundError("Final adapter not found.")

gc.collect()
torch.cuda.empty_cache()

merge_config = PreTrainedConfig.from_pretrained(
    checkpoint_dir
)
merge_config.device = "cpu"
merge_config.pretrained_path = BASE_MODEL_LOCAL
merge_config.use_peft = False

quiet_output = io.StringIO()

with (
    contextlib.redirect_stdout(quiet_output),
    contextlib.redirect_stderr(quiet_output),
):
    base_policy = SmolVLAPolicy.from_pretrained(
        BASE_MODEL_LOCAL,
        config=merge_config,
        strict=False,
    )

    peft_policy = PeftModel.from_pretrained(
        base_policy,
        checkpoint_dir,
        is_trainable=False,
        torch_device="cpu",
    )

    merged_policy = peft_policy.merge_and_unload(
        safe_merge=True
    )

shutil.rmtree(MERGED_MODEL_DIR, ignore_errors=True)
MERGED_MODEL_DIR.mkdir(parents=True, exist_ok=True)

merged_policy.config.use_peft = False
merged_policy.config.pretrained_path = None
merged_policy.config.push_to_hub = False
merged_policy.config.repo_id = None
merged_policy.config.device = None
merged_policy.config.load_vlm_weights = False
merged_policy.config.vlm_model_name = VLM_REPO

merged_policy.save_pretrained(MERGED_MODEL_DIR)

for pattern in [
    "policy_preprocessor.json",
    "policy_preprocessor*.safetensors",
    "policy_postprocessor.json",
    "policy_postprocessor*.safetensors",
]:
    for source_path in checkpoint_dir.glob(pattern):
        shutil.copy2(
            source_path,
            MERGED_MODEL_DIR / source_path.name,
        )

merged_weights_path = (
    MERGED_MODEL_DIR / "model.safetensors"
)

with safe_open(
    merged_weights_path,
    framework="pt",
    device="cpu",
) as weights:
    if any(
        "lora_" in key.lower()
        for key in weights.keys()
    ):
        raise RuntimeError(
            "LoRA parameters remain after merge."
        )

del peft_policy
del base_policy
del merged_policy

gc.collect()
torch.cuda.empty_cache()

print("Merged model ready.")

## 提出テンプレートへの組み込み と 本番 1mm ルールでの評価（最重要）

`lerobot-eval` による評価は **衝突判定を含まない**ため、ここでは行わない。
真の Track 1 スコア（1mm 判定つき成功率）は、配布キット側で測る。

### 1) 単一モデルとして使う
下のセルで作った zip を展開し、中身（`config.json` / `model.safetensors` /
`policy_preprocessor*` / `policy_postprocessor*`）を、キットの
`submission_template/model_weights/` 直下に置く。

### 2) アンサンブルの専門家として使う（推奨・(E)）
`MODEL_TAG`（と `TASK_NAME_FILTERS` / `SEED`）を変えて本ノートを複数回実行し、
それぞれの中身を別フォルダに置く:

```
submission_template/model_weights/
├── track1_lora/   { config.json, model.safetensors, ... }
├── bowl_expert/   { ... }
└── basket_expert/ { ... }
```

MyPolicy が全モデル × TTA ビューで予測して平均し、モデル間のばらつきを
不確実性として拾って自動で減速する。

### 3) 本番ルールで実測し、設定を最適化する（キット側で実行）

```bash
bash setup.sh && source env.sh          # 一度だけ
# 動作確認（1mm 判定つき成功率が出る）
python -m pipeline --server-url http://localhost:8000 --track track1 --n-episodes 5
# ↑ 別ターミナルで  python submission_template/policy_server.py  を起動しておく

# 推論設定（アンサンブル/ダンピング/速度等）を実測で最適化
python tune.py --tasks <task_id> --n-episodes 5
```

`tune.py` のランキング上位の設定を、`policy_server.py` のクラス変数に反映するか、
サーバー起動時に環境変数（`MYPOLICY_...`）で固定して提出する。


In [ ]:
from zipfile import ZIP_STORED, ZipFile

try:
    from google.colab import files
    _IN_COLAB = True
except Exception:
    _IN_COLAB = False

if MERGED_ZIP_PATH.exists():
    MERGED_ZIP_PATH.unlink()

with ZipFile(MERGED_ZIP_PATH, mode="w",
             compression=ZIP_STORED, allowZip64=True) as archive:
    for file_path in sorted(MERGED_MODEL_DIR.rglob("*")):
        if file_path.is_file():
            archive.write(
                file_path,
                arcname=Path(MERGED_MODEL_DIR.name)
                / file_path.relative_to(MERGED_MODEL_DIR),
            )

print(f"Saved: {MERGED_ZIP_PATH}")

# --- Google Drive へ自動コピー（1.5 でマウント済みなら実行される）---
# スマホ運用の想定経路: ここでコピーされたファイルを Drive アプリで
# 「共有可能なリンクを取得」し、そのリンクをチャットに貼るだけでよい。
if _HAS_DRIVE and DRIVE_MODEL_DIR is not None:
    DRIVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    drive_zip_path = DRIVE_MODEL_DIR / MERGED_ZIP_PATH.name
    shutil.copy2(MERGED_ZIP_PATH, drive_zip_path)
    print(f"Google Drive にも保存: {drive_zip_path}")
    print("→ Drive アプリでこのファイルを開き「リンクを共有」→"
          "「リンクを知っている全員」にしてURLを取得し、チャットに貼る")
else:
    print("→ 展開して中身を submission_template/model_weights/ (または "
          "model_weights/<MODEL_TAG>/) に置く")
    if _IN_COLAB:
        files.download(str(MERGED_ZIP_PATH))
